# 03 — Exportação e Deploy Mobile

Exporta os pesos treinados para os formatos usados no celular e prepara o PWA
`yoloinferApp` (ONNX Runtime Web) com o modelo da calçada.

In [ ]:
# exporta o best.pt pra ONNX (192 e 320) e tenta TFLite (opcional); guarda tudo em models/
from ultralytics import YOLO
import os, shutil, glob
os.makedirs('models', exist_ok=True)
model = YOLO('runs/calcada_v1/weights/best.pt')

exports = {}
exports['onnx_192'] = model.export(format='onnx', imgsz=192, simplify=True, opset=13)  # esse vai pro app
exports['onnx_320'] = model.export(format='onnx', imgsz=320, simplify=True, opset=13)  # versao maior
try:
    exports['tflite'] = model.export(format='tflite', imgsz=320)
except Exception as e:
    exports['tflite'] = f'opcional, não gerado: {e}'  # app usa ONNX, entao TFLite nao e obrigatorio
exports['pt'] = shutil.copy('runs/calcada_v1/weights/best.pt', 'models/best.pt')
for k, v in exports.items():
    print(f'{k:10s} -> {v}')

### Verificação: ONNX carrega e inferência funciona

In [ ]:
# testa o ONNX exportado rodando uma inferencia de verdade nas imagens de teste
import numpy as np
from ultralytics import YOLO
m = YOLO('models/calcada_192.onnx')
r = m.predict(source='data/dataset/images/test', imgsz=192, conf=0.25, save=True, project='runs', name='onnx_check')
print('classes:', r[0].names)
print('detectões na 1ª imagem:', len(r[0].boxes))

### Colocar o modelo no app

O app carrega `/models/calcada_192.onnx` como modelo padrão (entrada `1×3×192×192`,
mesmo tamanho do `INPUT_W/H` de `src/inference.js`).

In [ ]:
# copia o modelo pro diretorio publico do app, pra ele ja abrir com o modelo da calcada
app_models = 'app/yoloinferApp/Code/public/models'
os.makedirs(app_models, exist_ok=True)
shutil.copy('models/calcada_192.onnx', f'{app_models}/calcada_192.onnx')
print('copiado para', app_models)
print(os.listdir(app_models))

### Rodar no celular

```bash
cd app/yoloinferApp/Code
npm install
npm run dev     # desenvolvimento (HTTPS automático)
npm run build   # gera dist/ para publicar
```

1. publique o `dist/` em **HTTPS** (Netlify / GitHub Pages / túnel);
2. no celular: abrir no navegador → **Add to Home Screen** (PWA);
3. **Start camera** → a câmera traseira (`environment`) inicia e as caixas aparecem em tempo real;
4. use os sliders de *Confidence* e *IoU* e observe o medidor de FPS.

Sem build, clique em **Model** e selecione `models/calcada_192.onnx`.